In [ ]:
!pip install sentence-transformers chromadb groq pandas -q
print("completed ")

completed 


In [22]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
print("All libraries imported successfully")

All libraries imported successfully


In [ ]:
import os

GROQ_API_KEY = "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"
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized")
print("Note: If you see an error later, double check it")

Groq API client initialized
Note: If you see an error later, double check it


In [16]:
df=pd.read_csv('college_notes(1).csv')

print("Shape of database:",df.shape)

print("\n Column name:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape of database: (14, 4)

 Column name: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [19]:
print('subject in dataset')
print(df['subject'].value_counts())
print('sample of topics')
print(df[['note_id' , 'subject' , 'topic']].to_string(index=False))
print('length ')
df['content_length'] = df['content'].apply(len)
print(df[['topic' , 'content_length']].to_string(index=False))

subject in dataset
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    1
Name: count, dtype: int64
sample of topics
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python Program

In [18]:
documents = df['content'].tolist()
ids = [f'note_{row['note_id']}' for row in df.to_dict('records')]
metadatas = [
    {'subject': row['subject'], 'topic': row['topic']}
    for row in df.to_dict('records')
]
print("total chunks" , len(documents))
print("first column id" , {ids[2]})
print("first metadata" ,  metadatas[2])
print(f"first 100 words in document   {documents[2][:100]}...")

total chunks 14
first column id {'note_N003'}
first metadata {'subject': 'Data Engineering', 'topic': 'Data Cleaning'}
first 100 words in document   Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common c...


In [23]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
test_embedding = embedding_model.encode('this is a test sentence')
print(f'shape {test_embedding.shape}')
print(f"first 5 values od test embeddings {test_embedding[:5]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

shape (384,)
first 5 values od test embeddings [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [25]:
chroma_client=chromadb.Client()

collection=chroma_client.get_or_create_collection(name="college_notes_reg")

print("ChromaDB client created.")
print(f"collection name:{collection.name}")
print(f"Document in collection so far:{collection.count()}")

ChromaDB client created.
collection name:college_notes_reg
Document in collection so far:0


In [27]:
print("Genrating embedding for all 15 notes..")
print("This may take 15-30 sec...")
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape:{embeddings.shape}")

embeddings_list=embeddings.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)
print(f"\nDocument successfully added")
print(f"\nDocument in collection:{collection.count()}")

Genrating embedding for all 15 notes..
This may take 15-30 sec...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape:(14, 384)

Document successfully added

Document in collection:14


In [29]:
def retrive_relevant_chunks(question , top_k=3):
  """
  Given a user question , retrive the most relevant document chunks from chromaDB
  Parameters:
    question(str) : the user's question as a text string
    top_k  (int) : how many top results to return (default : 3)

  returns:
    a dictionary containing the relevant document , distance and metadata
  """
  question_embedding = embedding_model.encode(question).tolist()

  result = collection.query(
      query_embeddings=[question_embedding],

      n_results = top_k
  )
  return result

print("Retrieval function defind successfully!")
print("Function:retrive_relevent_chunks(question,top_k=3)")

Retrieval function defind successfully!
Function:retrive_relevent_chunks(question,top_k=3)


In [33]:
test_question="What is a ETL and how does it work in data engineering"
print(f"Test Question:{test_question}")
results=retrive_relevant_chunks(test_question,top_k=3)
print("\nTop 3 relivent chunks")

for i,(doc,disy,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"\nResult {i+1}")
    print(f"Subject:{meta['subject']}")
    print(f"Topic:{meta['topic']}")
    print(f"Distance:{disy}")

    print(f"Content:{doc[:120]}....")

Test Question:What is a ETL and how does it work in data engineering

Top 3 relivent chunks

Result 1
Subject:Data Engineering
Topic:ETL Pipelines
Distance:0.2014690637588501
Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i....

Result 2
Subject:Data Engineering
Topic:APIs and Data Collection
Distance:1.1242356300354004
Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ....

Result 3
Subject:Data Engineering
Topic:SQL Databases
Distance:1.3962396383285522
Content:A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interac....


In [1]:
def generate_rag_answer(question, context):
    """
    send the retrieves context and question to the groq llm for answer generation
    Parameters:
         question (str): the user's question
         context (str) : the retrieved context chunks( formatted string)
    Returns:
         answer (str) : the llm's generated message
    """
    system_prompt = """you are a helpfull academic assistant for engineerung students
    you will be given context retrieved from a college knowledge base , and a student's question
    RULES:
    1. Answer only using the information provided in the retrieved context.
    2. If the answer is not available in the context, clearly respond:
       "I could not find the answer in the provided knowledge base."
    3. Do not make up facts, assumptions, dates, or policies that are not present in the context.
    4. Provide clear, concise, and student-friendly explanations using simple language.
    5. When relevant, cite the section, document title, or source from which the answer was derived.
    """

    user_prompt = f"""context from knowledge base:
    {context}
    question: {question}
    answer the questions based only on the context provided.
    """

    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content
    return answer

print("RAG generation function defined")

RAG generation function defined
